# Test YOLO Inference on xView Dataset

This notebook:
1. Downloads the trained model from MinIO
2. Selects 5 random validation images
3. Runs inference and displays results

**Prerequisites:** Trained model in MinIO at `models/yolo26n_xview_best.pt`

## 1. Install Dependencies

In [ ]:
!pip install ultralytics minio pillow

## 2. Download Trained Model from MinIO

In [ ]:
import os
from minio import Minio
from pathlib import Path

# MinIO configuration
AWS_S3_ENDPOINT = os.getenv("AWS_S3_ENDPOINT", "minio-api-minio.apps.ocp.z8lpk.sandbox1242.opentlc.com")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minio123")
AWS_S3_BUCKET = os.getenv("AWS_S3_BUCKET", "demo")

endpoint = AWS_S3_ENDPOINT.replace('https://', '').replace('http://', '').rstrip('/')

client = Minio(
    endpoint,
    access_key=AWS_ACCESS_KEY_ID,
    secret_key=AWS_SECRET_ACCESS_KEY,
    secure=True
)

# Download trained model
model_path = Path("yolo26n_xview_best.pt")
object_name = "models/yolo26n_xview_best.pt"

if not model_path.exists():
    print(f"Downloading model from MinIO: {object_name}")
    client.fget_object(
        AWS_S3_BUCKET,
        object_name,
        str(model_path)
    )
    print(f"✓ Downloaded model ({model_path.stat().st_size / 1024 / 1024:.2f} MB)")
else:
    print(f"✓ Model already exists locally ({model_path.stat().st_size / 1024 / 1024:.2f} MB)")

## 3. Load Model and Select Random Images

In [ ]:
from ultralytics import YOLO
import random
from pathlib import Path

# Load the trained model
model = YOLO(str(model_path))
print(f"✓ Loaded model: {model_path}")

# Get validation images from the split file
val_split = Path("xView/images/autosplit_val.txt")

if val_split.exists():
    with open(val_split) as f:
        val_images = [line.strip() for line in f]
    
    print(f"\n✓ Found {len(val_images)} validation images")
    
    # Select 5 random images
    random_images = random.sample(val_images, min(5, len(val_images)))
    
    print(f"\nSelected {len(random_images)} random images for inference:")
    for i, img_path in enumerate(random_images, 1):
        img_name = Path(img_path).name
        print(f"  {i}. {img_name}")
else:
    print("✗ Validation split file not found!")
    print("  Make sure you've run prepare-xview-dataset.ipynb first")
    random_images = []

## 4. Run Inference and Display Results

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Set up matplotlib for better display
plt.rcParams['figure.figsize'] = (20, 4 * len(random_images))

fig, axes = plt.subplots(len(random_images), 1, figsize=(20, 6 * len(random_images)))
if len(random_images) == 1:
    axes = [axes]

for idx, img_path in enumerate(random_images):
    print(f"\n{'='*60}")
    print(f"Image {idx + 1}: {Path(img_path).name}")
    print(f"{'='*60}")
    
    # Run inference
    results = model(img_path, verbose=False)
    
    # Get the first result
    result = results[0]
    
    # Load original image
    img = Image.open(img_path)
    img_array = np.array(img)
    
    # Display image with detections
    ax = axes[idx]
    ax.imshow(img_array)
    ax.axis('off')
    
    # Count detections by class
    detections = {}
    
    if len(result.boxes) > 0:
        print(f"\nDetected {len(result.boxes)} objects:")
        
        # Draw bounding boxes
        for box in result.boxes:
            # Get box coordinates (xyxy format)
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            
            # Get class and confidence
            cls = int(box.cls[0].cpu().numpy())
            conf = float(box.conf[0].cpu().numpy())
            class_name = model.names[cls]
            
            # Count detections
            if class_name not in detections:
                detections[class_name] = 0
            detections[class_name] += 1
            
            # Draw rectangle
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2,
                edgecolor='red',
                facecolor='none'
            )
            ax.add_patch(rect)
            
            # Add label
            label = f"{class_name} {conf:.2f}"
            ax.text(
                x1, y1 - 5,
                label,
                color='white',
                fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7)
            )
        
        # Print detection summary
        for class_name, count in sorted(detections.items()):
            print(f"  - {class_name}: {count}")
    else:
        print("  No objects detected")
    
    # Set title
    title = f"{Path(img_path).name} - {len(result.boxes)} detections"
    ax.set_title(title, fontsize=14, pad=10)

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("Inference Complete!")
print(f"{'='*60}")

## 5. Performance Summary

In [ ]:
# Get inference speed statistics
print("\nInference Performance:")
print("="*40)

# Run inference on one image to get timing
if random_images:
    test_img = random_images[0]
    results = model(test_img, verbose=True)
    
    # Model info
    print(f"\nModel: {model.model_name}")
    print(f"Classes: {len(model.names)}")
    print(f"Image size: {model.overrides.get('imgsz', 640)}")

## Summary

This notebook demonstrated:
1. Loading a trained YOLO model from MinIO
2. Running inference on random validation images
3. Visualizing detection results with bounding boxes
4. Analyzing model performance

## Next Steps

- **Deploy the model** for real-time inference
- **Test on new satellite imagery** outside the validation set
- **Fine-tune** on specific object classes of interest
- **Export to ONNX** for optimized deployment